# WavMamba — WiFi-CSI HAR on UT-HAR & NTU-Fi

Public code accompanying the paper. Trains **WavMamba** — a multi-branch
(one branch per Haar DWT subband) CNN + bidirectional-Mamba model with adaptive
late fusion — for WiFi-CSI human activity recognition on **UT-HAR** and **NTU-Fi**.

Architecture (fixed): Haar `{HL, LH}` branches, attentive statistics pooling,
no stem GroupNorm, per-channel gate fusion.

## How to run

1. **Add Input** the two datasets (one containing `X_train.csv` for UT-HAR, one
   containing `train_amp/` for NTU-Fi), then set `Accelerator = GPU` and
   `Internet = On` in the notebook settings.
2. **Edit the Control panel cell below** — it holds every knob you might change
   (code URL, dataset paths, normalization flags, seeds, which ablations to run).
   Nothing else needs editing; the remaining cells just consume those values.
3. **Run all cells** top to bottom.

The defaults reproduce the paper protocol (`PRENORM=sensefi`, `Z_GRAN=pcb`,
UT-HAR merged val) with a single seed (42). Set `SEEDS = [0, 4, 8, 17, 42]` for
the 5-seed statistics.


In [ ]:
# Cell 1 — CONTROL PANEL: every knob for this run lives here.
# Edit this cell only; the cells below just consume these values. This cell runs
# BEFORE the code is cloned/installed, so it deliberately imports nothing from
# wavmamba — just plain assignments + fail-fast checks.

# --- Code source (the repo cloned in the next cell) --------------------------
REPO_URL = 'https://github.com/imhoangt/wavmamba.git'  # <- your repo
REPO_REF = None            # e.g. 'v1.0' or a commit SHA to pin; None = default branch HEAD

# --- Dataset locations (edit to match your Kaggle mounts) --------------------
# UT-HAR: dir that (recursively) contains X_train.csv / X_test.csv / ...
# NTU-Fi: dir that (recursively) contains train_amp/ and test_amp/.
# The loaders rglob for those markers, so the mount root or any parent works.
RAW_ROOT = {
    'uthar': '/kaggle/input/ut-har',
    'ntufi': '/kaggle/input/ntu-fi-har',
}

# --- What to run -------------------------------------------------------------
DATASETS = ['uthar', 'ntufi']   # drop one to run a single dataset
SEEDS    = [0, 4, 8, 17, 42]    # paper 5-seed protocol (the ablation table needs
                                # all 5: the paired per-seed test is the analysis,
                                # not mean +- std). Use [42] for a quick smoke run.
ABLATE   = True                 # also run the ablation sweep (Cell 6)
STUDY    = 'u'                  # which ladder, each filed under its own dir:
                                #   'a' -> outputs/ablation/    centre = full
                                #          two-branch WavMamba (paper model)
                                #   's' -> outputs/ablation_s/  centre = one
                                #          shared branch (WavMamba-S)
                                #   'u' -> outputs/ablation_u/  centre = uni-
                                #          Mamba, no backward pass (cost study)
# Only the rungs that still need training. The remaining rows of a re-centred
# ladder are configurations an earlier study ALREADY ran, so their runs are
# reused rather than repeated -- ablation.py fails the import if that ever stops
# being true. Listing them here would burn GPU AND produce a second, slightly
# different set of numbers for the same configs. Use 'all' only for study a.
#   study s: reuses center, c1_raw, c2_separate     (8 rungs left)
#   study u: reuses centre_u, u2_separate, u5_bimamba, u5_bilstm  (6 left)
# Already trained and unpacked into output/: study s (11/11) and study u's first
# 10 rungs. Only u5_unilstm is outstanding -- it completes the 2x2 of axis #5
# (Mamba vs LSTM) x (uni vs bi), so 'bilstm' stops moving two variables at once.
VARIANTS = ['u5_unilstm']

# --- Normalization / split (defaults = paper protocol) -----------------------
PRENORM   = 'sensefi'   # 'sensefi' = UT-HAR min-max / NTU-Fi (x-42.32)/4.98 | 'none'
Z_GRAN    = 'pcb'       # 'pcb' = per-channel-bin (C,F2), time collapsed | 'perpos' = (C,T2,F2)
MERGE_VAL = True        # UT-HAR only: True = test = X_test + X_val | False = test = X_test

# --- Where outputs go (usually leave as-is on Kaggle) ------------------------
OUT_ROOT = '/kaggle/working'

# --- Fail-fast validation (no wavmamba import yet) ---------------------------
assert PRENORM in ('none', 'sensefi'),  f'bad PRENORM {PRENORM!r}'
assert Z_GRAN  in ('perpos', 'pcb'),    f'bad Z_GRAN {Z_GRAN!r}'
assert DATASETS and set(DATASETS) <= {'uthar', 'ntufi'}, f'bad DATASETS {DATASETS!r}'
assert SEEDS, 'SEEDS must list at least one integer'
assert STUDY in ('a', 's', 'u'), f'bad STUDY {STUDY!r}'
print(f'PRENORM={PRENORM}  Z_GRAN={Z_GRAN}  MERGE_VAL={MERGE_VAL}')
print(f'DATASETS={DATASETS}  SEEDS={SEEDS}  ABLATE={ABLATE}')
print(f'STUDY={STUDY}  VARIANTS={VARIANTS}')
print(f'RAW_ROOT={RAW_ROOT}')


In [ ]:
# Cell 2 — Clone / update the public code from GitHub (uses REPO_URL from Cell 1)
import sys, subprocess
from pathlib import Path

CODE_PATH = Path('/kaggle/working/wavmamba')

if not CODE_PATH.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CODE_PATH)], check=True)
else:
    subprocess.run(['git', 'pull'], cwd=str(CODE_PATH), check=True)
if REPO_REF:
    subprocess.run(['git', 'fetch', '--depth', '1', 'origin', REPO_REF],
                   cwd=str(CODE_PATH), check=True)
    subprocess.run(['git', 'checkout', REPO_REF], cwd=str(CODE_PATH), check=True)

sys.path.insert(0, str(CODE_PATH))
print(f'Code at {CODE_PATH}  (ref={REPO_REF or "HEAD"})')


In [ ]:
# Cell 2b — Install dependencies + import wavmamba (run once per session)
#
# mamba-ssm / causal-conv1d ship prebuilt CUDA wheels that must match the torch
# C++ ABI, so install them with --no-build-isolation (build against the image's
# own torch instead of a fresh isolated one) + --no-deps (so pip does not
# re-resolve or replace an ABI-compatible torch).
import subprocess, sys
from pathlib import Path

def _pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', *args], check=True)

# Kaggle images already ship a CUDA torch; skip the explicit torch pin there.
# Uncomment to force a specific torch build locally:
# _pip('torch==2.7.0')
_pip('mamba-ssm', '--no-build-isolation', '--no-deps')
# causal-conv1d is OPTIONAL: it only provides a faster fused conv path. If its
# CUDA extension fails to build, mamba-ssm still runs (slower conv, identical
# results), so this one is best-effort and must not abort the cell.
try:
    _pip('causal-conv1d', '--no-build-isolation', '--no-deps')
except subprocess.CalledProcessError as e:
    print(f'[warn] causal-conv1d build failed ({e}); continuing without it '
          '(mamba-ssm falls back to its native conv path — same results).')
_pip('-r', str(CODE_PATH / 'requirements.txt'))

from wavmamba import (run, default_cfg, WavMamba, DIRMAP, bench_dirname,
                      build_bench, ABLATIONS, ablation_table, build_ablation_model)
from wavmamba.ablation import ABLATIONS_S, ABLATIONS_U, variant_front_end

# Resolve the study (from Cell 1) into the registry + output subdir it selects,
# then VARIANTS ('all' -> that registry) now that wavmamba is importable.
_STUDIES = {'a': (ABLATIONS,   'ablation'),
            's': (ABLATIONS_S, 'ablation_s'),
            'u': (ABLATIONS_U, 'ablation_u')}
REG, SUB = _STUDIES[STUDY]
VARIANTS = list(REG) if VARIANTS == 'all' else list(VARIANTS)
_bad = [v for v in VARIANTS if v not in REG]
assert not _bad, f'unknown variant(s) {_bad} for STUDY={STUDY!r}; choose from {list(REG)}'

# Per-dataset bench/output tag (merge_val is UT-HAR only) + a RAW_ROOT check.
def run_tag(ds):
    return bench_dirname(PRENORM, Z_GRAN, MERGE_VAL and ds == 'uthar')

print('Install + import OK : wavmamba')
print(f'  study={STUDY!r} -> outputs/{SUB}/  ({len(VARIANTS)} variant(s))')
for ds in DATASETS:
    ok = Path(RAW_ROOT[ds]).is_dir()
    print(f'  {ds:6s} tag={run_tag(ds):16s} raw={RAW_ROOT[ds]}  {"OK" if ok else "!! NOT FOUND"}')


In [ ]:
# Cell 4 — Smoke: WavMamba builds + forwards on GPU (fail-fast before build/train)
import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
# UT-HAR dims: n_antennas=3, f2=15 -> packed C = 2*3 = 6
_m = WavMamba(num_classes=7, n_antennas=3, f2=15).to(dev)
with torch.no_grad():
    _o = _m(torch.randn(2, 6, 125, 15, device=dev))   # T2=125 (250//2)
assert _o.shape == (2, 7), f'bad output {tuple(_o.shape)}'
print(f'SMOKE OK ({dev}) — WavMamba forward -> {tuple(_o.shape)}, '
      f'{sum(p.numel() for p in _m.parameters()):,} params')
del _m, _o
if dev == 'cuda': torch.cuda.empty_cache()


In [ ]:
# Cell 5 — Sweep: build + train each dataset (separate output dir per run)
import time, gc, torch

NL = chr(10)

def run_one(ds):
    raw   = RAW_ROOT[ds]
    # merge_val only exists for UT-HAR, so the tag is resolved PER DATASET —
    # bench_dirname() is the same helper build_bench() itself uses, so the
    # bench path and the output name always agree with the data.
    mv    = MERGE_VAL and ds == 'uthar'
    tag   = run_tag(ds)
    bench = Path(OUT_ROOT) / DIRMAP[ds] / 'bench' / tag
    out   = Path(f'{OUT_ROOT}/outputs/wavmamba_{ds}_{tag}')

    # Reuse an existing bench: /kaggle/working is wiped between sessions, but a
    # rerun inside one session should not pay the packing cost twice.
    if (bench / 'stats.json').exists():
        print(f'Bench exists, skipping build: {bench}')
    else:
        build_bench(ds, raw_root=raw, out_root=OUT_ROOT,
                    merge_val=mv, prenorm=PRENORM, z_gran=Z_GRAN)
    # run() reads classes / class names / dims from the bench's own stats.json.
    run(bench_dir=bench, output_dir=out, cfg=default_cfg(seeds=SEEDS),
        num_workers=4)
    return tag

results = {}
for ds in DATASETS:
    t0 = time.time()
    print(NL + '#' * 64 + NL + '#  wavmamba / ' + ds + ' / ' + run_tag(ds) + NL + '#' * 64)
    try:
        run_one(ds); results[ds] = 'OK'
    except Exception as e:
        results[ds] = f'FAILED: {type(e).__name__}: {e}'; print('!!', e)
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print(f"== {ds}: {results[ds]}  ({(time.time()-t0)/60:.1f} min)")
print(NL + '=== SWEEP SUMMARY ===')
for k, v in results.items():
    print(f'  {k:14s}: {v}')

In [ ]:
# Cell 6 — Ablation study (optional): one-variable-at-a-time sweep
#
# Runs the ablation VARIANTS (from Cell 1) for each dataset under the SAME
# protocol/flags as the main sweep. Each variant builds the bench it needs (the
# DWT bench, or a separate raw_ bench for a1_raw), trains an AblationWavMamba,
# and is filed under outputs/<SUB>/<ds>/<variant>/ where SUB comes from STUDY
# (Cell 1): 'ablation' for study a, 'ablation_s' for study s. Finished variants
# are skipped, so this cell is resumable -- and the study-s rungs reused from
# study a are skipped too if their runs were copied in. Set ABLATE = False in
# Cell 1 to skip it.
import json, time, gc, torch

NL = chr(10)

def study_base(ds):
    """Where this study's runs live. Defined once so the sweep and the table
    can never end up reading different layouts."""
    return Path(OUT_ROOT) / 'outputs' / SUB / ds

def ablate_one(ds):
    for variant in VARIANTS:
        fe  = variant_front_end(variant, REG)                  # 'dwt' | 'raw'
        mv  = MERGE_VAL and ds == 'uthar'
        tag = bench_dirname(PRENORM, Z_GRAN, mv, fe)
        bench = Path(OUT_ROOT) / DIRMAP[ds] / 'bench' / tag
        out   = study_base(ds) / variant

        mpath = out / 'metrics.json'
        if mpath.exists() and all(str(s) in json.load(open(mpath)).get('per_seed', {})
                                  for s in SEEDS):
            print(f'[skip] {variant}: already complete'); continue

        print(NL + '#' * 60 + NL + f'#  ablate / {ds} / {variant}  ({fe})' + NL + '#' * 60)
        if not (bench / 'stats.json').exists():
            build_bench(ds, raw_root=RAW_ROOT[ds], out_root=OUT_ROOT,
                        merge_val=mv, prenorm=PRENORM, z_gran=Z_GRAN, front_end=fe)
        meta = json.load(open(bench / 'stats.json'))['meta']
        run(bench_dir=bench, output_dir=out, cfg=default_cfg(seeds=SEEDS),
            num_workers=4,
            model_builder=lambda v=variant, m=meta: build_ablation_model(v, m, REG))

if ABLATE:
    for ds in DATASETS:
        t0 = time.time()
        try:
            ablate_one(ds)
        except Exception as e:
            print('!!', type(e).__name__, e)
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        print(f'== ablation {ds} done ({(time.time()-t0)/60:.1f} min)')
        print(NL + f'=== ABLATION TABLE ({SUB}) — {ds} ===')
        ablation_table(ds, base=study_base(ds), registry=REG)


In [ ]:
# Cell 7 — Package the outputs into a single .zip for download
#
# /kaggle/working is already exposed as the notebook Output (a directory tree),
# but a single archive is easier to grab in one click. This zips everything
# under outputs/ (both the main sweep and the ablation runs, including each
# variant's metrics.json + seeds/ + the ablation summary.md / summary.csv).
import shutil
from pathlib import Path

src = Path(OUT_ROOT) / 'outputs'
if src.is_dir():
    archive = shutil.make_archive(str(Path(OUT_ROOT) / 'wavmamba_outputs'),
                                  'zip', root_dir=str(src))
    mb = Path(archive).stat().st_size / 1e6
    print(f'Zipped -> {archive}  ({mb:.1f} MB)')
else:
    print(f'Nothing to zip: {src} does not exist yet (run the sweep cells first).')
